Hypothesis Testing, Statistical Significance, and using Scipy to run Student's t-tests

In [1]:
import numpy as np
import pandas as pd

from scipy import stats

In [2]:
df = pd.read_csv('datasets/grades.csv')
df.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
0,B73F2C11-70F0-E37D-8B10-1D20AFED50B1,92.733946,2015-11-02 06:55:34.282000000,83.030552,2015-11-09 02:22:58.938000000,67.164441,2015-11-12 08:58:33.998000000,53.011553,2015-11-16 01:21:24.663000000,47.710398,2015-11-20 13:24:59.692000000,38.168318,2015-11-22 18:31:15.934000000
1,98A0FAE0-A19A-13D2-4BB5-CFBFD94031D1,86.790821,2015-11-29 14:57:44.429000000,86.290821,2015-12-06 17:41:18.449000000,69.772657,2015-12-10 08:54:55.904000000,55.098125,2015-12-13 17:32:30.941000000,49.588313,2015-12-19 23:26:39.285000000,44.629482,2015-12-21 17:07:24.275000000
2,D0F62040-CEB0-904C-F563-2F8620916C4E,85.512541,2016-01-09 05:36:02.389000000,85.512541,2016-01-09 06:39:44.416000000,68.410033,2016-01-15 20:22:45.882000000,54.728026,2016-01-11 12:41:50.749000000,49.255224,2016-01-11 17:31:12.489000000,44.329701,2016-01-17 16:24:42.765000000
3,FFDF2B2C-F514-EF7F-6538-A6A53518E9DC,86.030665,2016-04-30 06:50:39.801000000,68.824532,2016-04-30 17:20:38.727000000,61.942079,2016-05-12 07:47:16.326000000,49.553663,2016-05-07 16:09:20.485000000,49.553663,2016-05-24 12:51:18.016000000,44.598297,2016-05-26 08:09:12.058000000
4,5ECBEEB6-F1CE-80AE-3164-E45E99473FB4,64.813800,2015-12-13 17:06:10.750000000,51.491040,2015-12-14 12:25:12.056000000,41.932832,2015-12-29 14:25:22.594000000,36.929549,2015-12-28 01:29:55.901000000,33.236594,2015-12-29 14:46:06.628000000,33.236594,2016-01-05 01:06:59.546000000


In [3]:
df.dtypes

student_id                    str
assignment1_grade         float64
assignment1_submission        str
assignment2_grade         float64
assignment2_submission        str
assignment3_grade         float64
assignment3_submission        str
assignment4_grade         float64
assignment4_submission        str
assignment5_grade         float64
assignment5_submission        str
assignment6_grade         float64
assignment6_submission        str
dtype: object

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              2315 non-null   str    
 1   assignment1_grade       2315 non-null   float64
 2   assignment1_submission  2315 non-null   str    
 3   assignment2_grade       2315 non-null   float64
 4   assignment2_submission  2315 non-null   str    
 5   assignment3_grade       2315 non-null   float64
 6   assignment3_submission  2315 non-null   str    
 7   assignment4_grade       2315 non-null   float64
 8   assignment4_submission  2315 non-null   str    
 9   assignment5_grade       2315 non-null   float64
 10  assignment5_submission  2315 non-null   str    
 11  assignment6_grade       2315 non-null   float64
 12  assignment6_submission  2315 non-null   str    
dtypes: float64(6), str(7)
memory usage: 235.2 KB


In [6]:
df.describe().round(2).loc[['min', 'max', 'mean']]

,assignment1_grade,assignment2_grade,assignment3_grade,assignment4_grade,assignment5_grade,assignment6_grade
min,14.42,12.98,12.31,9.13,8.21,7.39
max,100.70,99.94,99.66,98.76,97.57,97.57
mean,74.54,66.85,60.62,54.11,48.62,43.84


In [7]:
print(f"There are {df.shape[0]} rows and {df.shape[1]} columns")

There are 2315 rows and 13 columns


In [10]:
frames = ['assignment1_submission', 'assignment2_submission', 'assignment3_submission', 'assignment4_submission', 'assignment5_submission', 'assignment6_submission']

for frame in frames:
    df[frame] = pd.to_datetime(df[frame])

df.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
0,B73F2C11-70F0-E37D-8B10-1D20AFED50B1,92.733946,2015-11-02 06:55:34.282,83.030552,2015-11-09 02:22:58.938,67.164441,2015-11-12 08:58:33.998,53.011553,2015-11-16 01:21:24.663,47.710398,2015-11-20 13:24:59.692,38.168318,2015-11-22 18:31:15.934
1,98A0FAE0-A19A-13D2-4BB5-CFBFD94031D1,86.790821,2015-11-29 14:57:44.429,86.290821,2015-12-06 17:41:18.449,69.772657,2015-12-10 08:54:55.904,55.098125,2015-12-13 17:32:30.941,49.588313,2015-12-19 23:26:39.285,44.629482,2015-12-21 17:07:24.275
2,D0F62040-CEB0-904C-F563-2F8620916C4E,85.512541,2016-01-09 05:36:02.389,85.512541,2016-01-09 06:39:44.416,68.410033,2016-01-15 20:22:45.882,54.728026,2016-01-11 12:41:50.749,49.255224,2016-01-11 17:31:12.489,44.329701,2016-01-17 16:24:42.765
3,FFDF2B2C-F514-EF7F-6538-A6A53518E9DC,86.030665,2016-04-30 06:50:39.801,68.824532,2016-04-30 17:20:38.727,61.942079,2016-05-12 07:47:16.326,49.553663,2016-05-07 16:09:20.485,49.553663,2016-05-24 12:51:18.016,44.598297,2016-05-26 08:09:12.058
4,5ECBEEB6-F1CE-80AE-3164-E45E99473FB4,64.813800,2015-12-13 17:06:10.750,51.491040,2015-12-14 12:25:12.056,41.932832,2015-12-29 14:25:22.594,36.929549,2015-12-28 01:29:55.901,33.236594,2015-12-29 14:46:06.628,33.236594,2016-01-05 01:06:59.546


In [11]:
early = df[df['assignment1_submission'] < '2016']
early.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
0,B73F2C11-70F0-E37D-8B10-1D20AFED50B1,92.733946,2015-11-02 06:55:34.282,83.030552,2015-11-09 02:22:58.938,67.164441,2015-11-12 08:58:33.998,53.011553,2015-11-16 01:21:24.663,47.710398,2015-11-20 13:24:59.692,38.168318,2015-11-22 18:31:15.934
1,98A0FAE0-A19A-13D2-4BB5-CFBFD94031D1,86.790821,2015-11-29 14:57:44.429,86.290821,2015-12-06 17:41:18.449,69.772657,2015-12-10 08:54:55.904,55.098125,2015-12-13 17:32:30.941,49.588313,2015-12-19 23:26:39.285,44.629482,2015-12-21 17:07:24.275
4,5ECBEEB6-F1CE-80AE-3164-E45E99473FB4,64.813800,2015-12-13 17:06:10.750,51.491040,2015-12-14 12:25:12.056,41.932832,2015-12-29 14:25:22.594,36.929549,2015-12-28 01:29:55.901,33.236594,2015-12-29 14:46:06.628,33.236594,2016-01-05 01:06:59.546
5,D09000A0-827B-C0FF-3433-BF8FF286E15B,71.647278,2015-12-28 04:35:32.836,64.052550,2016-01-03 21:05:38.392,64.752550,2016-01-07 08:55:43.692,57.467295,2016-01-11 00:45:28.706,57.467295,2016-01-11 00:54:13.579,57.467295,2016-01-20 19:54:46.166
8,C9D51293-BD58-F113-4167-A7C0BAFCB6E5,66.595568,2015-12-25 02:29:28.415,52.916454,2015-12-31 01:42:30.046,48.344809,2016-01-05 23:34:02.180,47.444809,2016-01-02 07:48:42.517,37.955847,2016-01-03 21:27:04.266,37.955847,2016-01-19 15:24:31.060


In [12]:
late = df[df['assignment1_submission'] >= '2016']
late.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
2,D0F62040-CEB0-904C-F563-2F8620916C4E,85.512541,2016-01-09 05:36:02.389,85.512541,2016-01-09 06:39:44.416,68.410033,2016-01-15 20:22:45.882,54.728026,2016-01-11 12:41:50.749,49.255224,2016-01-11 17:31:12.489,44.329701,2016-01-17 16:24:42.765
3,FFDF2B2C-F514-EF7F-6538-A6A53518E9DC,86.030665,2016-04-30 06:50:39.801,68.824532,2016-04-30 17:20:38.727,61.942079,2016-05-12 07:47:16.326,49.553663,2016-05-07 16:09:20.485,49.553663,2016-05-24 12:51:18.016,44.598297,2016-05-26 08:09:12.058
6,3217BE3F-E4B0-C3B6-9F64-462456819CE4,87.498744,2016-03-05 11:05:25.408,69.998995,2016-03-09 07:29:52.405,55.999196,2016-03-16 22:31:24.316,50.399276,2016-03-18 07:19:26.032,45.359349,2016-03-19 10:35:41.869,45.359349,2016-03-23 14:02:00.987
7,F1CB5AA1-B3DE-5460-FAFF-BE951FD38B5F,80.576090,2016-01-24 18:24:25.619,72.518481,2016-01-27 13:37:12.943,65.266633,2016-01-30 14:34:36.581,65.266633,2016-02-03 22:08:49.002,65.266633,2016-02-16 14:22:23.664,65.266633,2016-02-18 08:35:04.796
9,E2C617C2-4654-622C-AB50-1550C4BE42A0,59.270882,2016-03-06 12:06:26.185,59.270882,2016-03-13 02:07:25.289,53.343794,2016-03-17 07:30:09.241,53.343794,2016-03-20 21:45:56.229,42.675035,2016-03-27 15:55:04.414,38.407532,2016-03-30 20:33:13.554


In [13]:
len(late)

1056

In [14]:
# Another method
late_new = df[~df.index.isin(early.index)]
len(late_new)

1056

In [16]:
print(early['assignment1_grade'].mean())
print(late['assignment1_grade'].mean())

74.94728457024304
74.0450648477065
